# Instructor Streaming — Live Extraction for Responsive UIs

**Week 1 | Notebook 3 of 4**

**What you'll learn:**
- `Partial[Model]` — streaming partial objects
- Building a live-updating extraction UI (Gradio)
- `create_iterable` — streaming lists of entities
- Async streaming — processing multiple documents in parallel
- Streaming from Anthropic vs OpenAI (API differences)
- Latency benchmarks: streaming vs. non-streaming UX

**Runtime:** ~45 minutes

In [1]:
# 💰 COST ESTIMATE
from src.cost_tracker import print_cost_warning

print_cost_warning("01_instructor/03_streaming.ipynb")

💰 COST ESTIMATE
----------------------------------------
Notebook:  01_instructor/03_streaming.ipynb
Task:      Streaming partial objects
Calls:     ~8

With GPT-4o:       $0.08 USD
With GPT-4o-mini:  $0.01 USD (10x cheaper)
With Ollama:       $0.00 USD (free, local)

💡 TIP: Set USE_SMALL_MODEL=true or USE_OLLAMA=true in .env to save money.
----------------------------------------


## 1. Setup

In [2]:
import instructor
from pydantic import BaseModel

from src.config import get_instructor_client, get_model

client = get_instructor_client()  # provider from LLM_PROVIDER in .env (default: openai)

## 2. Partial Streaming — Yields Incomplete Objects

In [3]:
class Order(BaseModel):
    customer: str
    items: list[str]
    total: float
    status: str


order_text = """
Order from John Doe: 2x MacBook Pro, 1x AirPods Pro.
Total: $4,298. Status: confirmed.
"""

print("Streaming partial objects:\n")
for partial_order in client.chat.completions.create_partial(
    model=get_model(),
    response_model=Order,
    messages=[{"role": "user", "content": f"Extract order: {order_text}"}],
):
    # Each iteration gives a progressively more complete object
    print(
        f"Customer: {partial_order.customer or '...'} | Items: {len(partial_order.items or [])} | Total: {partial_order.total or '...'}"
    )

Streaming partial objects:

Customer: ... | Items: 0 | Total: ...
Customer: ... | Items: 0 | Total: ...
Customer: ... | Items: 0 | Total: ...
Customer: ... | Items: 0 | Total: ...
Customer: John | Items: 0 | Total: ...
Customer: John Doe | Items: 0 | Total: ...
Customer: John Doe | Items: 0 | Total: ...
Customer: John Doe | Items: 0 | Total: ...
Customer: John Doe | Items: 1 | Total: ...
Customer: John Doe | Items: 1 | Total: ...
Customer: John Doe | Items: 1 | Total: ...
Customer: John Doe | Items: 1 | Total: ...
Customer: John Doe | Items: 1 | Total: ...
Customer: John Doe | Items: 1 | Total: ...
Customer: John Doe | Items: 2 | Total: ...
Customer: John Doe | Items: 2 | Total: ...
Customer: John Doe | Items: 2 | Total: ...
Customer: John Doe | Items: 2 | Total: ...
Customer: John Doe | Items: 2 | Total: ...
Customer: John Doe | Items: 2 | Total: ...
Customer: John Doe | Items: 2 | Total: ...
Customer: John Doe | Items: 2 | Total: ...
Customer: John Doe | Items: 2 | Total: ...
Custome

## 3. Streaming Lists — Extract Multiple Items Progressively

In [4]:
class LineItem(BaseModel):
    product_name: str
    quantity: int
    unit_price: float


invoice_text = """
Invoice #1234
- MacBook Pro x2 @ $1,999 each
- AirPods Pro x1 @ $249
- USB-C Cable x3 @ $19 each
"""

print("Streaming individual line items:\n")
for item in client.chat.completions.create_iterable(
    model=get_model(),
    response_model=LineItem,
    messages=[{"role": "user", "content": f"Extract all line items: {invoice_text}"}],
):
    print(f"  Got: {item.product_name} x{item.quantity} @ ${item.unit_price}")

Streaming individual line items:

  Got: MacBook Pro x2 @ $1999.0
  Got: AirPods Pro x1 @ $249.0
  Got: USB-C Cable x3 @ $19.0


## 4. Live-Updating Extraction UI with Gradio

In [5]:
import gradio as gr


def extract_with_streaming(text: str):
    """Stream extraction results to Gradio UI."""

    class ExtractedInfo(BaseModel):
        name: str | None = None
        company: str | None = None
        amount: float | None = None
        date: str | None = None

    result_text = ""
    for partial in client.chat.completions.create_partial(
        model=get_model(),
        response_model=ExtractedInfo,
        messages=[{"role": "user", "content": f"Extract info from: {text}"}],
    ):
        result_text = partial.model_dump_json(indent=2)
        yield result_text


# Create Gradio interface
demo = gr.Interface(
    fn=extract_with_streaming,
    inputs=gr.Textbox(label="Input Text", value="John from Acme Corp paid $500 on 2025-01-15"),
    outputs=gr.Textbox(label="Extracted Info"),
    title="Live Extraction with Instructor Streaming",
    description="Type text and watch the structured data fill in live!",
)

# Uncomment to launch:
demo.launch()

print("✅ Gradio demo ready. Uncomment demo.launch() to start the UI.")

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


✅ Gradio demo ready. Uncomment demo.launch() to start the UI.


## 5. Async Streaming — Processing Multiple Documents

In [6]:
import asyncio

from src.config import get_async_openai_client, get_model

async_client = instructor.from_openai(get_async_openai_client())


class User(BaseModel):
    name: str
    age: int
    city: str


async def extract_user(text: str) -> User:
    return await async_client.chat.completions.create(
        model=get_model(),
        response_model=User,
        messages=[{"role": "user", "content": text}],
    )


texts = [
    "Alice, 30, lives in New York",
    "Bob is 25 and from San Francisco",
    "Carol, 35, based in Chicago",
]


async def process_batch():
    tasks = [extract_user(t) for t in texts]
    results = await asyncio.gather(*tasks)
    return results


results = await process_batch()
for user in results:
    print(f"{user.name}, {user.age}, {user.city}")

Alice, 30, New York
Bob, 25, San Francisco
Carol, 35, Chicago


## 6. Streaming API Differences: OpenAI vs Anthropic

In [7]:
# The ONLY difference between providers is how the client is created.
# The extraction code below is literally the same function for both.


class Person(BaseModel):
    name: str
    age: int
    city: str


def stream_extract(client, model: str, text: str):
    """Provider-agnostic streaming extraction — same code for everyone."""
    for partial in client.chat.completions.create_partial(
        model=model,
        response_model=Person,
        messages=[{"role": "user", "content": text}],
        max_tokens=1024,  # required by Anthropic; OpenAI accepts it too
    ):
        yield partial


text = "Alice, 30, lives in New York"

# Different clients, different models — same call site.
# Providers without a key in .env are skipped, so the cell always runs.
results = {}
for provider in ("openai", "anthropic", "gemini", "groq"):
    try:
        c = get_instructor_client(provider)
        results[provider] = list(stream_extract(c, get_model(provider), text))[-1]
        print(
            f"{provider.capitalize()} ({get_model(provider)}): {results[provider].model_dump_json()}"
        )
    except ValueError as e:
        print(f"{provider.capitalize()}: skipped — no API key in .env")

# Same schema in, same structured objects out — regardless of provider
signatures = {
    p: {k: type(v).__name__ for k, v in r.model_dump().items()} for p, r in results.items()
}
assert len(set(map(frozenset, signatures.values()))) <= 1, f"Mismatched outputs: {signatures}"
print(
    "\n✅ Identical structured output — Instructor abstracts streaming differences between providers"
)
print("   Same code works for OpenAI, Anthropic, Gemini, Groq, and local models")

OpenAI (gpt-4o):      {"name":"Alice","age":30,"city":"New York"}
Anthropic (claude-opus-4-6): {"name":"Alice","age":30,"city":"New York"}

✅ Identical structured output — Instructor abstracts streaming differences between providers
   Same code works for OpenAI, Anthropic, and local models


## 7. Latency Benchmark: Streaming vs Non-Streaming UX

In [7]:
import time


class Summary(BaseModel):
    title: str
    key_points: list[str]


long_text = "Artificial intelligence is transforming..." * 50  # Long text

# Non-streaming: wait for everything
start = time.time()
summary = client.chat.completions.create(
    model=get_model(),
    response_model=Summary,
    messages=[{"role": "user", "content": f"Summarize: {long_text[:500]}"}],
)
non_stream_time = time.time() - start

# Streaming: first field appears immediately
start = time.time()
first_field_time = None
for partial in client.chat.completions.create_partial(
    model=get_model(),
    response_model=Summary,
    messages=[{"role": "user", "content": f"Summarize: {long_text[:500]}"}],
):
    if partial.title and first_field_time is None:
        first_field_time = time.time() - start
        break

print(f"Non-streaming total time: {non_stream_time:.2f}s")
print(f"Streaming — first field visible: {first_field_time:.2f}s")
print(
    f"\n💡 UX improvement: users see data {non_stream_time / first_field_time:.1f}x faster with streaming"
)

Non-streaming total time: 1.79s
Streaming — first field visible: 1.10s

💡 UX improvement: users see data 1.6x faster with streaming
